# LinearAttention

经典的 scaled dot-product attention 的计算方式是：

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

在这个过程中，假设 $Q,K,V$ 的 shape 都是：

$$
Q,K,V \in \mathbb{R}^{N \times d}
$$

其中 $N$ 是 sequence length，$d$ 是 hidden dimension 或 head dimension。那么 $QK^T$ 的 shape 是：

$$
QK^T \in \mathbb{R}^{N \times N}
$$

计算 $QK^T$ 的时间复杂度是：

$$
O(N^2d)
$$

softmax 作用在 $N \times N$ 的 attention score 上，复杂度是：

$$
O(N^2)
$$

最后再乘以 $V$，复杂度也是：

$$
O(N^2d)
$$

因此，当序列长度 $N$ 很大时，标准 attention 的主要开销来自 $N \times N$ 的 attention map，整体复杂度通常可以记为：

$$
O(N^2d)
$$

如果只关心 sequence length 带来的增长，也常常简写成：

$$
O(N^2)
$$

这也是标准 attention 在长序列场景下开销很大的原因。

## 优化 Attention 计算

这里，优化时间复杂度的最大阻碍在于 softmax。假如暂时忽略 softmax 和 $\sqrt{d_k}$ 这个常数缩放，attention 可以写成：

$$
O = QK^TV
$$

根据矩阵乘法的结合律，可以先计算 $K^TV$，再和 $Q$ 相乘：

$$
O = Q(K^TV)
$$

其中：

$$
K^TV: [d,N] \times [N,d] \rightarrow [d,d]
$$

这一步复杂度是：

$$
O(Nd^2)
$$

然后：

$$
Q(K^TV): [N,d] \times [d,d] \rightarrow [N,d]
$$

这一步复杂度也是：

$$
O(Nd^2)
$$

如果 $d$ 远小于 $N$，那么复杂度就从和 $N^2$ 相关，变成了和 $N$ 近似线性相关。这就是 Linear Attention 想要利用的核心思想。

不过，标准 attention 不能简单地直接去掉 softmax。因为 softmax 至少带来了两个重要性质：

1. 归一化。每个 query 对所有 key 的 attention weights 之和为 1，所以输出不会随着序列长度 $N$ 无限增长。
2. 非负性。softmax 后的 attention weights 都是非负的，不会出现大量正负值互相抵消导致结果不稳定的问题。

因此，如果想把 attention 写成线性形式，就需要用别的方式替代 softmax，同时尽量保留“非负”和“归一化”这两个性质。

## Linear Attention 形式

Linear Attention 常见思路是把原来的相似度：

$$
\exp\left(\frac{q_i^T k_j}{\sqrt{d_k}}\right)
$$

替换成一个可以分解的 kernel feature map：

$$
sim(q_i,k_j)=\phi(q_i)^T\phi(k_j)
$$

其中 $\phi(\cdot)$ 通常会被设计成非负映射。这样第 $i$ 个 token 的输出可以写成：

$$
o_i = \frac{\sum_{j=1}^{N}\phi(q_i)^T\phi(k_j)v_j}{\sum_{j=1}^{N}\phi(q_i)^T\phi(k_j)}
$$

把它改写成矩阵形式，就可以先计算和所有 key/value 相关的部分：

$$
S = \phi(K)^T V
$$

再计算归一化项：

$$
z_i = \phi(q_i)^T \sum_{j=1}^{N}\phi(k_j)
$$

最终：

$$
o_i = \frac{\phi(q_i)^T S}{z_i}
$$

这样就避免了显式构造 $N \times N$ 的 attention matrix。

## 几种处理方式

如果直接去掉 softmax，会带来一系列问题：

1. 没有归一化操作。如果 $N$ 不断增长，attention 的结果可能也会随着序列长度增长，导致结果不稳定。
2. attention score 可能有负值，最后不同位置的结果可能正负抵消。

对于问题一，可以通过分母归一化项解决；对于问题二，可以通过非负 feature map 解决。

论文《Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention》的处理方式是对 $Q$ 和 $K$ 做一个非负映射，例如：

$$
\phi(x)=\operatorname{elu}(x)+1
$$

这样可以保证 $\phi(Q)$ 和 $\phi(K)$ 的值非负。

《Efficient Attention: Attention with Linear Complexities》的思路则是把 softmax 拆开近似处理，例如计算：

$$
\operatorname{softmax}_d(Q)\left(\operatorname{softmax}_N(K)^T V\right)
$$

其中 $\operatorname{softmax}_d(Q)$ 沿 feature 维度归一化，$\operatorname{softmax}_N(K)$ 沿 sequence 维度归一化。这样做可以避免直接生成完整的 $N \times N$ attention matrix，同时让结果保持非负和一定程度的归一化。

此外，还有一些从稀疏化或近似计算角度加速 attention 的方式，例如 Reformer、Sparse Attention 等。它们和 Linear Attention 的目标类似，都是为了缓解长序列下标准 attention 的二次复杂度问题，但具体方法不同。



References: 苏剑林科学空间，线性 Attention 的探索
